# TN1 phần B — TCN và DS-TCN

Chạy **song song** với `TN1_LSTM.ipynb` ở một phiên Colab khác. Hai notebook độc lập hoàn toàn, không cần chờ nhau.

| notebook | chạy gì | thời gian |
|---|---|---|
| TN1_LSTM.ipynb | LSTM: 4 fold CV, rồi 3 seed test GHIJ | ~3 giờ |
| **TN1_TCN_DSTCN.ipynb** ← đang mở | TCN-64 và DS-TCN-64: 4 fold CV mỗi cái | ~2.2 giờ |
| TN1.ipynb | gộp kết quả, so sánh, chạy GHIJ cho kiến trúc thắng | ~1.5 giờ |

## Hai kiến trúc

| model | kênh | tham số | so LSTM |
|---|---|---|---|
| TCN | 64 | 151.513 | −90% |
| DS-TCN | 64 | 56.281 | −96% |

Khác nhau **đúng một chỗ** — phép tích chập trong mỗi khối:

```python
# TCN thường
nn.Conv1d(64, 64, kernel_size=3, dilation=d)

# DS-TCN, tách làm hai bước (Howard et al. 2017, mục 3.1)
nn.Conv1d(64, 64, 3, dilation=d, groups=64)   # depthwise: mỗi kênh một bộ lọc
nn.Conv1d(64, 64, 1)                          # pointwise: chỉ trộn kênh
```

Mọi thứ khác giống hệt: `kernel=3`, `n_blocks=6`, hai tầng conv mỗi khối, `dropout=0.0`, BatchNorm, ReLU, nối tắt. Trích dẫn từng tham số ở [`docs/THAM_CHIEU.md`](../docs/THAM_CHIEU.md).

## Giao thức

Bốn fold cố định trên tám người `A B C D E F K L`, dùng y nguyên cho mọi thí nghiệm:

```
val_AB   train C D E F K L    chấm A B
val_CE   train A B D F K L    chấm C E
val_DF   train A B C E K L    chấm D F
val_KL   train A B C D E F    chấm K L
```

Cấu hình giữ nguyên như MobiVital công bố: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr_threshold` 0.9, không RevIN. Một seed.

Điểm chấm trên **buổi ghi thô**, model tự chọn kênh, không nhìn nhịp thở thật.


## 1. Chuẩn bị Colab


Mount Drive để lấy lại cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


Tải mã nguồn rồi vào thư mục đó. `setup_colab.py` clone MobiVital và ghim commit `4319731d` — `src/mobivital_reference.py` mượn sáu hàm từ repo họ.


In [2]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : b17df2e
commit MobiVital : 4319731 (đã ghim)
GPU              : NVIDIA L4, 23034 MiB


Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB — chỉ train trên cửa sổ đã cắt, chấm trên `by_user/*.npz`.


In [3]:
!python scripts/restore_processed_data_on_drive.py


by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


## 2. TCN-64 — 4 fold CV

Khoảng **1 giờ**.


In [4]:
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64


thực nghiệm tn1  -> runs/tn1/
cấu hình tcn_c64_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03438  pearson 0.4631   0.6 phút
epoch  1  mse 0.02013  pearson 0.5452   1.1 phút
epoch  2  mse 0.01884  pearson 0.5694   1.7 phút
epoch  3  mse 0.01795  pearson 0.5822   2.2 phút
epoch  4  mse 0.01734  pearson 0.5904   2.7 phút
epoch  5  mse 0.01677  pearson 0.5989   3.3 phút
epoch  6  mse 0.01640  pearson 0.6046   3.8 phút
epoch  7  mse 0.01600  pearson 0.6108   4.3 phút
epoch  8  mse 0.01567  pearson 0.6154   4.9 phút
epoch  9  mse 0.01537  pearson 0.6196   5.4 phút
epoch 10  mse 0.01515  pearson 0.6235   6.0 phút
epoch 11  mse 0.01484  pearson 0.6271   6.5 phút
epoch 12  mse 0.01459  pearson 0.6300   7.1 phút
epoch 13  mse 0.01434  pearson 0.6323   7.6 phút
epoch 14  mse 0.01410  pearson 0.6348   8.1 phút
epoch 15  mse 0.01389  pearson 0.6373   8.7 phút
epoch 16  mse 0.01365

## 3. DS-TCN-64 — 4 fold CV

Khoảng **1.2 giờ**.


In [5]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64


thực nghiệm tn1  -> runs/tn1/
cấu hình ds_tcn_c64_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.05256  pearson 0.4147   0.6 phút
epoch  1  mse 0.02089  pearson 0.5300   1.2 phút
epoch  2  mse 0.01917  pearson 0.5617   1.8 phút
epoch  3  mse 0.01828  pearson 0.5788   2.5 phút
epoch  4  mse 0.01759  pearson 0.5897   3.1 phút
epoch  5  mse 0.01710  pearson 0.5973   3.7 phút
epoch  6  mse 0.01675  pearson 0.6037   4.4 phút
epoch  7  mse 0.01641  pearson 0.6088   5.0 phút
epoch  8  mse 0.01614  pearson 0.6127   5.6 phút
epoch  9  mse 0.01592  pearson 0.6169   6.2 phút
epoch 10  mse 0.01567  pearson 0.6204   6.9 phút
epoch 11  mse 0.01549  pearson 0.6227   7.5 phút
epoch 12  mse 0.01528  pearson 0.6255   8.1 phút
epoch 13  mse 0.01509  pearson 0.6275   8.7 phút
epoch 14  mse 0.01493  pearson 0.6290   9.4 phút
epoch 15  mse 0.01478  pearson 0.6310   10.0 phút
epoch 16  mse 0.0

## 4. Xem nhanh

So hai kiến trúc với nhau. LSTM chưa có ở phiên này nên chưa so được với mốc — việc đó làm ở `TN1_final_evaluation.ipynb` sau khi gộp.


In [6]:
!python scripts/compare_cv.py --experiment tn1



BẢNG 1 — cv_score, thực nghiệm tn1
cấu hình                            tham số   cv_score    cv_std   từng fold
----------------------------------------------------------------------------------------------------
ds_tcn_c64_mse_corr0.9_seed0          56281   0.741376  0.065523   AB 0.7793  CE 0.7857  DF 0.6282  KL 0.7723
tcn_c64_mse_corr0.9_seed0            151513   0.737742  0.085114   AB 0.7847  CE 0.7888  DF 0.5903  KL 0.7872

cv_score = trung bình điểm macro của 4 fold. Điểm macro = trung bình theo
NGƯỜI, không theo buổi ghi — mỗi người có số buổi khác nhau.



## 5. Cất kết quả

Nén ra tên riêng `tn1_tcn.zip` để **không đè** tệp của phiên chạy LSTM.


In [7]:
!cd runs && zip -qr /content/drive/MyDrive/mobivital/tn1_tcn.zip tn1 summary.csv
!ls -la /content/drive/MyDrive/mobivital/


total 2729552
-rw------- 1 root root 2640629760 Sep  3 15:32 by_user.tar
-rw------- 1 root root    5643034 Sep  4 00:57 tn0.zip
-rw------- 1 root root   39092589 Sep  4 16:57 tn1_lstm.zip
-rw------- 1 root root    3241215 Sep  4 16:58 tn1_tcn.zip
-rw------- 1 root root  106452993 Sep  3 15:29 windows.tar.gz
